In [56]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

GERACAO_TOOLS = """ 
Você é um assistente que ajuda a construir catálogos de ferramentas (tools) para testar sistemas de IA com function calling.

Gere exatamente 30 tools (ferramentas) fictícias, porém realistas, que representem problemas reais do dia a dia — cobrindo uma ampla variedade de domínios (produtividade, agenda, comunicação, finanças, saúde, viagens, e-commerce, desenvolvimento de software, casa inteligente, análise de dados, clima, etc.), de forma que o catálogo seja interessante e sirva como destaque em um projeto de portfólio.

Para cada tool, gere os seguintes campos:
- nome: identificador único em snake_case, em inglês, descrevendo uma ação clara (ex: schedule_meeting, get_stock_price)
- tipo: a categoria/domínio da tool (ex: "agenda", "financas", "clima")
- descricao: uma frase clara e específica do que a função faz, escrita como documentação real de uma API
- parametros: lista de parâmetros que a função recebe, cada um com: nome, tipo (string, integer, number, boolean ou array) e se é obrigatório (true/false)
- dificuldade: "facil" ou "dificil" — distribua de forma equilibrada (aproximadamente 15 fáceis e 15 difíceis). Considere "facil" quando os parâmetros são poucos e diretos; "dificil" quando há mais parâmetros, alguma ambiguidade, ou exigem inferência de contexto.

Retorne a resposta em formato JSON, como uma lista de 30 objetos, cada um seguindo exatamente essa estrutura, sem nenhum texto fora do JSON.
"""

resposta = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": GERACAO_TOOLS}],
)

print(resposta.choices[0].message.content)


```
[
  {
    "nome": "schedule_meeting",
    "tipo": "agenda",
    "descricao": "Agenda uma reunião com base nos horários disponíveis dos participantes.",
    "parametros": [
      { "nome": "title", "tipo": "string", "obrigatorio": true },
      { "nome": "start_time", "tipo": "string", "obrigatorio": true },
      { "nome": "end_time", "tipo": "string", "obrigatorio": true },
      { "nome": "participants", "tipo": "array", "obrigatorio": true }
    ],
    "dificuldade": "dificil"
  },
  {
    "nome": "get_stock_price",
    "tipo": "financas",
    "descricao": "Retorna o preço atual de uma ação específica.",
    "parametros": [
      { "nome": "stock_symbol", "tipo": "string", "obrigatorio": true }
    ],
    "dificuldade": "facil"
  },
  {
    "nome": "send_email",
    "tipo": "comunicacao",
    "descricao": "Envia um e-mail para um destinatário específico.",
    "parametros": [
      { "nome": "recipient", "tipo": "string", "obrigatorio": true },
      { "nome": "subject", "tipo":

In [57]:
from pydantic import BaseModel
from typing import Literal
import json

class Parametro(BaseModel):
    nome: str
    tipo: Literal["string", "integer", "number", "boolean", "array"]
    obrigatorio: bool

class Tool(BaseModel):
    nome: str
    tipo: str
    descricao: str
    parametros: list[Parametro]
    dificuldade: Literal["facil", "dificil"]

texto = resposta.choices[0].message.content.strip()
texto = texto.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

dados = json.loads(texto)
tools_catalogo = [Tool.model_validate(item) for item in dados]

print(len(tools_catalogo))

34


In [58]:
queries_geradas = {"texto_query": [],
                   "nome_tool": [],
                   "tipo_tool": [],
                   "funcao_tool": [],
                   "dificuldade_query": []}

indice_tools = {}
for cada_tool in tools_catalogo: 
    indice_tools[cada_tool.nome] = cada_tool
print(indice_tools, end="")

{'schedule_meeting': Tool(nome='schedule_meeting', tipo='agenda', descricao='Agenda uma reunião com base nos horários disponíveis dos participantes.', parametros=[Parametro(nome='title', tipo='string', obrigatorio=True), Parametro(nome='start_time', tipo='string', obrigatorio=True), Parametro(nome='end_time', tipo='string', obrigatorio=True), Parametro(nome='participants', tipo='array', obrigatorio=True)], dificuldade='dificil'), 'get_stock_price': Tool(nome='get_stock_price', tipo='financas', descricao='Retorna o preço atual de uma ação específica.', parametros=[Parametro(nome='stock_symbol', tipo='string', obrigatorio=True)], dificuldade='facil'), 'send_email': Tool(nome='send_email', tipo='comunicacao', descricao='Envia um e-mail para um destinatário específico.', parametros=[Parametro(nome='recipient', tipo='string', obrigatorio=True), Parametro(nome='subject', tipo='string', obrigatorio=True), Parametro(nome='body', tipo='string', obrigatorio=True)], dificuldade='facil'), 'track_p

In [59]:
tools_texto = ""

for texto in tools_catalogo:
    cada_tool = f'nome:{texto.nome}, descrição: {texto.descricao}, parâmetros: {texto.parametros}, dificuldade: {texto.dificuldade}\n'
    tools_texto += cada_tool
print(tools_texto)


regra_facil = "a query deve mencionar explicitamente todos os parâmetros que a tool precisa, de forma direta"
regra_dificil = "a query deve ser indireta, vaga, ou omitir algum parâmetro, forçando inferência"

prompt_query = f"""
Você está gerando dados de treinamento para um sistema de tool use (function calling).

Existem duas categorias de dificuldade:
- "facil": {regra_facil}
- "dificil": {regra_dificil}

Abaixo está o catálogo completo de tools disponíveis, cada uma já marcada com sua dificuldade:
{tools_texto}

Para CADA uma dessas tools, invente exatamente 13 perguntas de usuário, em português, realistas e variadas entre si, que uma pessoa faria a um assistente e que só poderiam ser respondidas chamando aquela tool específica — seguindo a regra de dificuldade indicada para ela.

As perguntas devem soar naturais, como se fossem escritas por uma pessoa de verdade — não mencione o nome técnico da tool na pergunta, apenas descreva a necessidade em linguagem natural.

Retorne a resposta em formato JSON, como uma lista de objetos, cada um com "nome_tool" (o nome da tool) e "queries" (a lista das 13 perguntas geradas para ela), sem nenhum texto fora do JSON.
"""


resposta = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt_query}],
    max_completion_tokens=9000,
)

print(resposta.choices[0].message.content)

nome:schedule_meeting, descrição: Agenda uma reunião com base nos horários disponíveis dos participantes., parâmetros: [Parametro(nome='title', tipo='string', obrigatorio=True), Parametro(nome='start_time', tipo='string', obrigatorio=True), Parametro(nome='end_time', tipo='string', obrigatorio=True), Parametro(nome='participants', tipo='array', obrigatorio=True)], dificuldade: dificil
nome:get_stock_price, descrição: Retorna o preço atual de uma ação específica., parâmetros: [Parametro(nome='stock_symbol', tipo='string', obrigatorio=True)], dificuldade: facil
nome:send_email, descrição: Envia um e-mail para um destinatário específico., parâmetros: [Parametro(nome='recipient', tipo='string', obrigatorio=True), Parametro(nome='subject', tipo='string', obrigatorio=True), Parametro(nome='body', tipo='string', obrigatorio=True)], dificuldade: facil
nome:track_package, descrição: Retorna o status de entrega de um pacote., parâmetros: [Parametro(nome='tracking_number', tipo='string', obrigato

In [60]:
class ToolQueries(BaseModel):
    nome_tool: str
    queries: list[str] 

texto = resposta.choices[0].message.content.strip()
texto = texto.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

dados_queries = json.loads(texto)
queries_catalogo = [ToolQueries.model_validate(item) for item in dados_queries]

print(queries_catalogo)
print(len(dados_queries))

[ToolQueries(nome_tool='schedule_meeting', queries=['Preciso agendar uma reunião com o time de marketing para discutir o novo projeto.', 'Como posso encontrar um horário que funcione para todos os participantes da reunião?', 'Quais são os horários disponíveis para a reunião da semana que vem?', 'Preciso remarcar a reunião de hoje para amanhã.', 'Como faço para agendar uma reunião com pessoas de diferentes fusos horários?', 'Quais são os melhores horários para agendar reuniões com os clientes?', 'Preciso agendar uma reunião com o CEO para discutir o relatório financeiro.', 'Como posso evitar conflitos de horários com outras reuniões?', 'Quais são os horários mais produtivos para agendar reuniões?', 'Preciso agendar uma reunião com um grupo grande de pessoas.', 'Como faço para agendar uma reunião com alguém que está em férias?', 'Quais são os melhores dias da semana para agendar reuniões?', 'Preciso agendar uma reunião com um cliente que está em um fuso horário diferente.']), ToolQueries

In [61]:
for resultado in queries_catalogo:
    tool = indice_tools[resultado.nome_tool]

    for pergunta in resultado.queries:
        queries_geradas["texto_query"].append(pergunta)
        queries_geradas["nome_tool"].append(tool.nome)
        queries_geradas["tipo_tool"].append(tool.tipo)
        queries_geradas["funcao_tool"].append(tool.descricao)
        queries_geradas["dificuldade_query"].append(tool.dificuldade)

print(len(queries_geradas["texto_query"]))

377


In [62]:
import pandas as pd 

#dados das queries
df = pd.DataFrame(queries_geradas)

#salvar o arquivo
df.to_csv('queries_geradas.csv', index=False)
